# Create Patients Table (Prospective Data Pull)

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
icu_file_path = '/home/lu.kuang/thesis/Prospective_DataCleaning/updated_cleaning/DataMerged_Prospective_0430_updated.xlsx'

In [3]:
edw_file_path = '/home/lu.kuang/thesis/Prospective_DataCleaning/updated_cleaning/Lee_EDW.xlsx'

In [4]:
crosswalk = '/home/lu.kuang/thesis/Prospective_DataCleaning/updated_cleaning/MRN_ULI.xlsx'

In [5]:
PATIENTS = pd.read_excel(io = icu_file_path, sheet_name = 'PATIENT VISIT')
PATIENTS = PATIENTS[['RECORD_ID','MRN','GENDER','DOB', 'ICU_ADMIT', 'ICU_DISCHARGE', 'HOSP_ADMIT','HOSP_DISCHARGE','ICU_SURVIVAL','HOSP_SURVIVAL']]
PATIENTS

,RECORD_ID,MRN,GENDER,DOB,ICU_ADMIT,ICU_DISCHARGE,HOSP_ADMIT,HOSP_DISCHARGE,ICU_SURVIVAL,HOSP_SURVIVAL
0,189844,1005018831,M,1963-06-25,2021-08-01 02:14:00,2021-08-03 14:08:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A
1,189946,1001596889,M,1949-09-14,2021-08-03 03:54:00,2021-08-05 20:51:00,2021-08-03 03:54:00,2021-08-11 13:51:00,A,A
2,190055,1005018831,M,1963-06-25,2021-08-04 23:56:00,2021-08-07 18:05:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A
3,190072,1036362778,M,1972-05-05,2021-08-05 06:52:00,2021-08-12 17:23:00,2021-08-05 06:52:00,2021-08-12 17:23:00,D,D
4,190106,1002625000,M,1974-06-22,2021-08-05 20:09:00,2021-08-07 23:29:00,2021-08-05 20:09:00,2021-08-10 18:20:00,A,A
...,...,...,...,...,...,...,...,...,...,...
347,209687,1042419836,F,1976-03-01,2022-07-18 16:33:00,2022-07-21 15:35:00,2022-07-18 16:33:00,2022-07-24 11:25:00,A,A
348,209746,1013229958,F,1961-03-08,2022-07-19 23:23:00,2022-07-23 18:23:00,2022-07-19 23:23:00,2022-08-02 17:24:00,A,A
349,209792,1041163179,M,1983-09-02,2022-07-20 22:52:00,2022-07-23 07:20:00,2022-07-20 22:52:00,2022-07-23 07:20:00,D,D
350,209795,1013129836,M,1993-10-07,2022-07-21 00:22:00,2022-07-24 19:15:00,2022-07-21 00:22:00,2022-07-24 19:15:00,A,A


In [6]:
# Ensure datetime columns are parsed
PATIENTS['ICU_DISCHARGE'] = pd.to_datetime(PATIENTS['ICU_DISCHARGE'], errors='coerce')
PATIENTS['HOSP_DISCHARGE'] = pd.to_datetime(PATIENTS['HOSP_DISCHARGE'], errors='coerce')

def determine_dod(row):
    icu_survived = row['ICU_SURVIVAL'] == 'A'
    hosp_survived = row['HOSP_SURVIVAL'] == 'A'

    if icu_survived and not hosp_survived:
        return row['HOSP_DISCHARGE']
    elif not icu_survived:
        if pd.isna(row['ICU_DISCHARGE']) and pd.isna(row['HOSP_DISCHARGE']):
            return pd.NaT
        elif pd.isna(row['ICU_DISCHARGE']):
            return row['HOSP_DISCHARGE']
        elif pd.isna(row['HOSP_DISCHARGE']):
            return row['ICU_DISCHARGE']
        else:
            return min(row['ICU_DISCHARGE'], row['HOSP_DISCHARGE'])
    else:
        return pd.NaT  # patient survived both

# Apply logic
PATIENTS['DOD1'] = PATIENTS.apply(determine_dod, axis=1)
PATIENTS

,RECORD_ID,MRN,GENDER,DOB,ICU_ADMIT,ICU_DISCHARGE,HOSP_ADMIT,HOSP_DISCHARGE,ICU_SURVIVAL,HOSP_SURVIVAL,DOD1
0,189844,1005018831,M,1963-06-25,2021-08-01 02:14:00,2021-08-03 14:08:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,NaT
1,189946,1001596889,M,1949-09-14,2021-08-03 03:54:00,2021-08-05 20:51:00,2021-08-03 03:54:00,2021-08-11 13:51:00,A,A,NaT
2,190055,1005018831,M,1963-06-25,2021-08-04 23:56:00,2021-08-07 18:05:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,NaT
3,190072,1036362778,M,1972-05-05,2021-08-05 06:52:00,2021-08-12 17:23:00,2021-08-05 06:52:00,2021-08-12 17:23:00,D,D,2021-08-12 17:23:00
4,190106,1002625000,M,1974-06-22,2021-08-05 20:09:00,2021-08-07 23:29:00,2021-08-05 20:09:00,2021-08-10 18:20:00,A,A,NaT
...,...,...,...,...,...,...,...,...,...,...,...
347,209687,1042419836,F,1976-03-01,2022-07-18 16:33:00,2022-07-21 15:35:00,2022-07-18 16:33:00,2022-07-24 11:25:00,A,A,NaT
348,209746,1013229958,F,1961-03-08,2022-07-19 23:23:00,2022-07-23 18:23:00,2022-07-19 23:23:00,2022-08-02 17:24:00,A,A,NaT
349,209792,1041163179,M,1983-09-02,2022-07-20 22:52:00,2022-07-23 07:20:00,2022-07-20 22:52:00,2022-07-23 07:20:00,D,D,2022-07-23 07:20:00
350,209795,1013129836,M,1993-10-07,2022-07-21 00:22:00,2022-07-24 19:15:00,2022-07-21 00:22:00,2022-07-24 19:15:00,A,A,NaT


# 1. Add Height

In [7]:
HEIGHT_WEIGHT = pd.read_excel(io = icu_file_path, sheet_name = 'HEIGHT_WEIGHT')
HEIGHT_WEIGHT

,RECORD_ID,SITE,ICU_ADMIT,Admit_weight_kg,discharge_weight_kg,Admit_height_cm,discharge_height_cm
0,189844,FMC ICU,2021-08-01 02:14:00,165.2,165.0,178.0,178.0
1,189946,FMC ICU,2021-08-03 03:54:00,103.9,103.9,183.0,183.0
2,190055,FMC ICU,2021-08-04 23:56:00,171.5,165.2,178.0,178.0
3,190072,FMC ICU,2021-08-05 06:52:00,78.4,78.9,185.0,185.0
4,190106,FMC ICU,2021-08-05 20:09:00,159.1,159.1,NaN,NaN
...,...,...,...,...,...,...,...
295,210081,FMC ICU,2022-07-27 12:26:00,77.7,78.0,186.0,186.0
296,210086,FMC ICU,2022-07-27 15:29:00,82.7,82.7,NaN,NaN
297,210095,FMC ICU,2022-07-27 18:00:00,112.2,112.2,165.0,165.0
298,210098,FMC ICU,2022-07-27 20:04:00,99.0,99.0,187.0,187.0


In [8]:
def HEIGHT(df):
    a_height = df['Admit_height_cm']
    d_height = df['discharge_height_cm']
    
    if ~np.isnan(a_height) and np.isnan(d_height):
        height = a_height
    elif np.isnan(a_height) and ~np.isnan(d_height):
        height = d_height
    elif ~np.isnan(a_height) and ~np.isnan(d_height):
        height = np.mean([a_height,d_height])
    else:
        height = None
    return height
HEIGHT_WEIGHT['HEIGHT'] = HEIGHT_WEIGHT.apply(lambda x: HEIGHT(x), axis = 1)
HEIGHT_WEIGHT.drop(['Admit_height_cm','discharge_height_cm'], inplace = True, axis = 1)
HEIGHT_WEIGHT.rename(columns={'record_id':'RECORD_ID'}, inplace = True)
# Adding height info to PATIENTS
PATIENTS = PATIENTS.merge(right = HEIGHT_WEIGHT[['RECORD_ID','HEIGHT']], on = ['RECORD_ID'], how = 'left')
PATIENTS

,RECORD_ID,MRN,GENDER,DOB,ICU_ADMIT,ICU_DISCHARGE,HOSP_ADMIT,HOSP_DISCHARGE,ICU_SURVIVAL,HOSP_SURVIVAL,DOD1,HEIGHT
0,189844,1005018831,M,1963-06-25,2021-08-01 02:14:00,2021-08-03 14:08:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,NaT,178.0
1,189946,1001596889,M,1949-09-14,2021-08-03 03:54:00,2021-08-05 20:51:00,2021-08-03 03:54:00,2021-08-11 13:51:00,A,A,NaT,183.0
2,190055,1005018831,M,1963-06-25,2021-08-04 23:56:00,2021-08-07 18:05:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,NaT,178.0
3,190072,1036362778,M,1972-05-05,2021-08-05 06:52:00,2021-08-12 17:23:00,2021-08-05 06:52:00,2021-08-12 17:23:00,D,D,2021-08-12 17:23:00,185.0
4,190106,1002625000,M,1974-06-22,2021-08-05 20:09:00,2021-08-07 23:29:00,2021-08-05 20:09:00,2021-08-10 18:20:00,A,A,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
347,209687,1042419836,F,1976-03-01,2022-07-18 16:33:00,2022-07-21 15:35:00,2022-07-18 16:33:00,2022-07-24 11:25:00,A,A,NaT,157.0
348,209746,1013229958,F,1961-03-08,2022-07-19 23:23:00,2022-07-23 18:23:00,2022-07-19 23:23:00,2022-08-02 17:24:00,A,A,NaT,164.0
349,209792,1041163179,M,1983-09-02,2022-07-20 22:52:00,2022-07-23 07:20:00,2022-07-20 22:52:00,2022-07-23 07:20:00,D,D,2022-07-23 07:20:00,177.0
350,209795,1013129836,M,1993-10-07,2022-07-21 00:22:00,2022-07-24 19:15:00,2022-07-21 00:22:00,2022-07-24 19:15:00,A,A,NaT,174.0


# 2. Add Vital Status

In [9]:
VITALSTAT = pd.read_excel(io = edw_file_path, sheet_name = 'VITALSTAT')
CW = pd.read_excel(io = crosswalk)
VITALSTAT = pd.merge(VITALSTAT, CW, on='PHN_ENC', how='inner')
VITALSTAT

,PHN_ENC,AGE,SEX,DETHDATE,HOSP_ID,PL_DETH,AUTOPSY,U_CAUSE,MRN,ULI,ULIN,CHARTNO
0,59611892680,83,2,2022-06-02,71.0,1,N,C760,1045491030,596401500,596401500,1045491030
1,131217061762,65,2,2022-06-30,16.0,1,N,I489,1024470732,941434490,941434490,1024470732
2,36214911243,52,1,2021-09-03,16.0,1,N,K704,1032726083,256113130,256113130,1032726083
3,101113250479,76,2,2022-04-14,16.0,1,N,I489,1003124284,640742210,640742210,1003124284
4,45517001572,56,1,2022-05-18,16.0,1,N,R99,1004152037,817587230,817587230,1004152037
...,...,...,...,...,...,...,...,...,...,...,...,...
98,316605331778,73,1,2022-05-06,16.0,1,N,I219,1045569702,226527400,226527400,1045569702
99,151723260251,60,1,2022-05-14,16.0,1,N,I269,1045590294,709206120,709206120,1022944688
100,154428781393,69,2,2022-06-09,16.0,1,N,U071,1023551789,219599810,219599810,1023551789
101,7419142971,65,1,2022-12-02,NaN,2,N,C329,1017927086,562801421,562801421,1017927086


In [10]:
VITALSTAT.rename(columns={'DETHDATE':'DOD2',
                     'PL_DETH' : 'DEATH_PLACE',
                     'U_CAUSE' : 'DEATH_CAUSE'}, inplace = True)
VITALSTAT

,PHN_ENC,AGE,SEX,DOD2,HOSP_ID,DEATH_PLACE,AUTOPSY,DEATH_CAUSE,MRN,ULI,ULIN,CHARTNO
0,59611892680,83,2,2022-06-02,71.0,1,N,C760,1045491030,596401500,596401500,1045491030
1,131217061762,65,2,2022-06-30,16.0,1,N,I489,1024470732,941434490,941434490,1024470732
2,36214911243,52,1,2021-09-03,16.0,1,N,K704,1032726083,256113130,256113130,1032726083
3,101113250479,76,2,2022-04-14,16.0,1,N,I489,1003124284,640742210,640742210,1003124284
4,45517001572,56,1,2022-05-18,16.0,1,N,R99,1004152037,817587230,817587230,1004152037
...,...,...,...,...,...,...,...,...,...,...,...,...
98,316605331778,73,1,2022-05-06,16.0,1,N,I219,1045569702,226527400,226527400,1045569702
99,151723260251,60,1,2022-05-14,16.0,1,N,I269,1045590294,709206120,709206120,1022944688
100,154428781393,69,2,2022-06-09,16.0,1,N,U071,1023551789,219599810,219599810,1023551789
101,7419142971,65,1,2022-12-02,NaN,2,N,C329,1017927086,562801421,562801421,1017927086


In [11]:
# Adding DOD info (VS) to PATIENTS
PATIENTS = PATIENTS.merge(right = VITALSTAT, on = ['MRN'], how = 'left')
PATIENTS

,RECORD_ID,MRN,GENDER,DOB,ICU_ADMIT,ICU_DISCHARGE,HOSP_ADMIT,HOSP_DISCHARGE,ICU_SURVIVAL,HOSP_SURVIVAL,...,AGE,SEX,DOD2,HOSP_ID,DEATH_PLACE,AUTOPSY,DEATH_CAUSE,ULI,ULIN,CHARTNO
0,189844,1005018831,M,1963-06-25,2021-08-01 02:14:00,2021-08-03 14:08:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,189946,1001596889,M,1949-09-14,2021-08-03 03:54:00,2021-08-05 20:51:00,2021-08-03 03:54:00,2021-08-11 13:51:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,190055,1005018831,M,1963-06-25,2021-08-04 23:56:00,2021-08-07 18:05:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,190072,1036362778,M,1972-05-05,2021-08-05 06:52:00,2021-08-12 17:23:00,2021-08-05 06:52:00,2021-08-12 17:23:00,D,D,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,190106,1002625000,M,1974-06-22,2021-08-05 20:09:00,2021-08-07 23:29:00,2021-08-05 20:09:00,2021-08-10 18:20:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,209687,1042419836,F,1976-03-01,2022-07-18 16:33:00,2022-07-21 15:35:00,2022-07-18 16:33:00,2022-07-24 11:25:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
358,209746,1013229958,F,1961-03-08,2022-07-19 23:23:00,2022-07-23 18:23:00,2022-07-19 23:23:00,2022-08-02 17:24:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
359,209792,1041163179,M,1983-09-02,2022-07-20 22:52:00,2022-07-23 07:20:00,2022-07-20 22:52:00,2022-07-23 07:20:00,D,D,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
360,209795,1013129836,M,1993-10-07,2022-07-21 00:22:00,2022-07-24 19:15:00,2022-07-21 00:22:00,2022-07-24 19:15:00,A,A,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
# Creating a single DOD based on DOD1 and DOD2
PATIENTS['DOD'] = PATIENTS.apply(lambda x:
                                x['DOD2']
                                if pd.isnull((x['DOD1']))
                                else 
                                x['DOD1'], 
                                axis = 1)
PATIENTS.drop(['DOD1','DOD2'], inplace = True, axis = 1)
PATIENTS

,RECORD_ID,MRN,GENDER,DOB,ICU_ADMIT,ICU_DISCHARGE,HOSP_ADMIT,HOSP_DISCHARGE,ICU_SURVIVAL,HOSP_SURVIVAL,...,AGE,SEX,HOSP_ID,DEATH_PLACE,AUTOPSY,DEATH_CAUSE,ULI,ULIN,CHARTNO,DOD
0,189844,1005018831,M,1963-06-25,2021-08-01 02:14:00,2021-08-03 14:08:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
1,189946,1001596889,M,1949-09-14,2021-08-03 03:54:00,2021-08-05 20:51:00,2021-08-03 03:54:00,2021-08-11 13:51:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
2,190055,1005018831,M,1963-06-25,2021-08-04 23:56:00,2021-08-07 18:05:00,2021-08-01 02:14:00,2021-08-21 11:37:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3,190072,1036362778,M,1972-05-05,2021-08-05 06:52:00,2021-08-12 17:23:00,2021-08-05 06:52:00,2021-08-12 17:23:00,D,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2021-08-12 17:23:00
4,190106,1002625000,M,1974-06-22,2021-08-05 20:09:00,2021-08-07 23:29:00,2021-08-05 20:09:00,2021-08-10 18:20:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
357,209687,1042419836,F,1976-03-01,2022-07-18 16:33:00,2022-07-21 15:35:00,2022-07-18 16:33:00,2022-07-24 11:25:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
358,209746,1013229958,F,1961-03-08,2022-07-19 23:23:00,2022-07-23 18:23:00,2022-07-19 23:23:00,2022-08-02 17:24:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
359,209792,1041163179,M,1983-09-02,2022-07-20 22:52:00,2022-07-23 07:20:00,2022-07-20 22:52:00,2022-07-23 07:20:00,D,D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2022-07-23 07:20:00
360,209795,1013129836,M,1993-10-07,2022-07-21 00:22:00,2022-07-24 19:15:00,2022-07-21 00:22:00,2022-07-24 19:15:00,A,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [13]:
PATIENTS = PATIENTS.groupby(by = 'MRN', axis = 0)
PATIENTS = PATIENTS.agg({    
    'GENDER' : lambda x: list(set(x))[0],
    'HEIGHT' : lambda x: np.round(x.median(),0),
    'DOB' : lambda x: list(set(x))[0],
    'DOD' : lambda x: x.max(),
    'HOSP_ID' : lambda x: list(set(x))[0],
    'ULI': lambda x: list(set(x))[0],
    'PHN_ENC': lambda x: list(set(x))[0],
    'CHARTNO': lambda x: list(set(x))[0],
    'DEATH_PLACE' : lambda x: list(set(x))[0],
    'DEATH_CAUSE' : lambda x: list(set(x))[0],
}).reset_index()

/tmp/ipykernel_3732400/4281757745.py:1: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  PATIENTS = PATIENTS.groupby(by = 'MRN', axis = 0)


In [14]:
PATIENTS

,MRN,GENDER,HEIGHT,DOB,DOD,HOSP_ID,ULI,PHN_ENC,CHARTNO,DEATH_PLACE,DEATH_CAUSE
0,1000292019,M,175.0,1943-05-19,2022-07-10 00:00:00,423.0,103714300.0,2.282225e+11,1.000292e+09,5.0,C719
1,1000545531,M,180.0,1953-06-29,2022-06-16 22:22:00,NaN,NaN,NaN,NaN,NaN,NaN
2,1000552495,F,168.0,1949-12-06,2022-04-22 23:24:00,16.0,234773110.0,9.320552e+10,1.000552e+09,1.0,M329
3,1000558914,M,179.0,1947-06-24,2022-06-16 16:03:00,16.0,897759800.0,2.434261e+11,1.000559e+09,1.0,C19
4,1001072881,M,182.0,1975-10-03,NaT,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
271,1045724828,M,188.0,1971-07-06,NaT,NaN,NaN,NaN,NaN,NaN,NaN
272,1045724851,M,174.0,1955-02-24,NaT,NaN,NaN,NaN,NaN,NaN,NaN
273,1045738455,M,175.0,1946-05-20,NaT,NaN,NaN,NaN,NaN,NaN,NaN
274,1045739768,M,186.0,2000-05-26,NaT,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Creating PATIENT_ID and renaming ROW_ID
PATIENTS = PATIENTS.reset_index()
PATIENTS.rename(columns={'index':'PATIENT_ID'}, inplace = True)
PATIENTS['PATIENT_ID'] = PATIENTS['PATIENT_ID'].apply(lambda x: 'p'+str(x).rjust(5, '0'))
PATIENTS.index.set_names(['ROW_ID'], inplace = True)
PATIENTS = PATIENTS[['PATIENT_ID', 'MRN', 'CHARTNO', 'ULI', 'PHN_ENC', 'GENDER', 'HEIGHT', 'DOB', 'DOD', 'HOSP_ID',
                     'DEATH_PLACE', 'DEATH_CAUSE']]
PATIENTS

,PATIENT_ID,MRN,CHARTNO,ULI,PHN_ENC,GENDER,HEIGHT,DOB,DOD,HOSP_ID,DEATH_PLACE,DEATH_CAUSE
ROW_ID,,,,,,,,,,,,
0,P00000,1000292019,1.000292e+09,103714300.0,2.282225e+11,M,175.0,1943-05-19,2022-07-10 00:00:00,423.0,5.0,C719
1,P00001,1000545531,NaN,NaN,NaN,M,180.0,1953-06-29,2022-06-16 22:22:00,NaN,NaN,NaN
2,P00002,1000552495,1.000552e+09,234773110.0,9.320552e+10,F,168.0,1949-12-06,2022-04-22 23:24:00,16.0,1.0,M329
3,P00003,1000558914,1.000559e+09,897759800.0,2.434261e+11,M,179.0,1947-06-24,2022-06-16 16:03:00,16.0,1.0,C19
4,P00004,1001072881,NaN,NaN,NaN,M,182.0,1975-10-03,NaT,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
271,P00271,1045724828,NaN,NaN,NaN,M,188.0,1971-07-06,NaT,NaN,NaN,NaN
272,P00272,1045724851,NaN,NaN,NaN,M,174.0,1955-02-24,NaT,NaN,NaN,NaN
273,P00273,1045738455,NaN,NaN,NaN,M,175.0,1946-05-20,NaT,NaN,NaN,NaN


In [16]:
PATIENTS.to_csv('PATIENTS.csv', sep = ";", encoding = 'Latin-1')